# Intermediate Analysis — Questions 6-10

**Techniques:** `CTE`, `WINDOW FUNCTIONS`, `DATE functions`, `NTILE`, `RANK`

| # | Question |
|---|----------|
| Q6  | Monthly revenue trend — best and worst months |
| Q7  | Actual vs estimated delivery time gap |
| Q8  | Top 3 best-selling products per category |
| Q9  | Top 10% sellers — share of total revenue |
| Q10 | Weekday vs weekend order behavior |

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.db_utils import run_query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## Q6: Monthly Revenue Trend

In [ ]:
q6 = run_query("""
    WITH monthly_revenue AS (
        SELECT
            STRFTIME('%Y-%m', o.order_purchase_timestamp) AS month,
            ROUND(SUM(op.payment_value), 2)               AS revenue
        FROM orders o
        JOIN order_payments op ON o.order_id = op.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY month
    )
    SELECT month, revenue,
           RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
    FROM monthly_revenue
    ORDER BY month
""")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(q6['month'], q6['revenue'], marker='o', color='steelblue', linewidth=2)
ax.fill_between(q6['month'], q6['revenue'], alpha=0.15, color='steelblue')

best = q6.loc[q6['revenue_rank'] == 1]
ax.annotate(f"Best: {best['month'].values[0]}\n${best['revenue'].values[0]:,.0f}",
            xy=(best.index[0], best['revenue'].values[0]),
            xytext=(best.index[0] - 3, best['revenue'].values[0] * 0.92),
            arrowprops=dict(arrowstyle='->', color='green'),
            fontsize=9, color='green')

plt.xticks(rotation=45, ha='right')
ax.set_title('Monthly Revenue Trend (Delivered Orders)', fontsize=14, fontweight='bold')
ax.set_ylabel('Revenue (BRL)')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('../data/q6_monthly_revenue.png', bbox_inches='tight')
plt.show()
q6.head(5)

## Q7: Actual vs Estimated Delivery Time

In [ ]:
q7 = run_query("""
    SELECT
        ROUND(AVG(JULIANDAY(order_delivered_customer_date) - JULIANDAY(order_purchase_timestamp)), 1) AS avg_actual_days,
        ROUND(AVG(JULIANDAY(order_estimated_delivery_date) - JULIANDAY(order_purchase_timestamp)), 1) AS avg_estimated_days,
        ROUND(AVG(JULIANDAY(order_estimated_delivery_date) - JULIANDAY(order_delivered_customer_date)), 1) AS avg_days_early_late,
        COUNT(*) AS total_orders
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
""")

print('Delivery Performance Summary')
print('='*40)
print(f"Avg actual delivery:    {q7['avg_actual_days'].values[0]} days")
print(f"Avg estimated delivery: {q7['avg_estimated_days'].values[0]} days")
early_late = q7['avg_days_early_late'].values[0]
label = 'days EARLY on average' if early_late > 0 else 'days LATE on average'
print(f"Result: {abs(early_late)} {label}")
print(f"Total orders analyzed:  {q7['total_orders'].values[0]:,}")
q7

## Q8: Top 3 Products per Category

In [ ]:
q8 = run_query("""
    WITH ranked_products AS (
        SELECT
            t.product_category_name_english AS category,
            oi.product_id,
            COUNT(oi.order_id) AS total_sold,
            RANK() OVER (
                PARTITION BY t.product_category_name_english
                ORDER BY COUNT(oi.order_id) DESC
            ) AS rank_in_category
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        JOIN product_category_name_translation t
            ON p.product_category_name = t.product_category_name
        GROUP BY category, oi.product_id
    )
    SELECT * FROM ranked_products
    WHERE rank_in_category <= 3
    ORDER BY category, rank_in_category
""")

print(f'Total categories: {q8["category"].nunique()}')
print(f'Total records: {len(q8)}')
q8.head(15)

## Q9: Top 10% Sellers — Revenue Concentration

In [ ]:
q9 = run_query("""
    WITH seller_revenue AS (
        SELECT seller_id,
               ROUND(SUM(price), 2) AS total_revenue,
               NTILE(10) OVER (ORDER BY SUM(price) DESC) AS decile
        FROM order_items
        GROUP BY seller_id
    ),
    totals AS (SELECT SUM(total_revenue) AS grand_total FROM seller_revenue)
    SELECT
        CASE WHEN decile = 1 THEN 'Top 10%' ELSE 'Bottom 90%' END AS seller_group,
        COUNT(*) AS seller_count,
        ROUND(SUM(total_revenue), 2) AS group_revenue,
        ROUND(SUM(total_revenue) * 100.0 / MAX(grand_total), 2) AS revenue_share_pct
    FROM seller_revenue, totals
    GROUP BY seller_group
""")

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(q9['group_revenue'], labels=q9['seller_group'],
       autopct='%1.1f%%', colors=['#e74c3c', '#95a5a6'], startangle=90)
ax.set_title('Revenue Concentration: Top 10% vs Bottom 90% Sellers',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/q9_seller_concentration.png', bbox_inches='tight')
plt.show()
q9

## Q10: Weekday vs Weekend Order Behavior

In [ ]:
q10 = run_query("""
    SELECT
        CASE
            WHEN CAST(STRFTIME('%w', order_purchase_timestamp) AS INTEGER) IN (0, 6)
            THEN 'Weekend' ELSE 'Weekday'
        END AS day_type,
        COUNT(*) AS total_orders,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_orders
    FROM orders
    GROUP BY day_type
""")

fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(data=q10, x='day_type', y='total_orders', ax=ax,
            palette=['#3498db', '#e67e22'])
for i, row in q10.iterrows():
    ax.text(i, row['total_orders'] + 200, f"{row['pct_of_orders']}%",
            ha='center', fontweight='bold')
ax.set_title('Weekday vs Weekend Orders', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Total Orders')
plt.tight_layout()
plt.savefig('../data/q10_weekday_weekend.png', bbox_inches='tight')
plt.show()
q10